In [0]:
-- ===========================================================
-- Databricks SQL variables (use ${var} for expansion)
-- ===========================================================
DECLARE OR REPLACE VARIABLE calendaryear INT;
DECLARE OR REPLACE VARIABLE calendarmonth INT;
SET VAR calendaryear = 2026;
SET VAR calendarmonth = 1;

-- ===========================================================
-- Combined Query
-- ===========================================================
WITH 
----------------------------------------------------------------
-- Common: Branch Names
----------------------------------------------------------------
branchnames AS (
  SELECT backendname, alias
  FROM `na-bu-cdm-dev`.`staging_usdwh_tkefinance`.`epmdimensionmetadata`
),

----------------------------------------------------------------
-- Query 1: FTE / Headcount (preserves your original logic)
----------------------------------------------------------------
alldata AS (
  SELECT
    CASE 
      WHEN CostType = 'COS DIRECT'   THEN 'Direct'
      WHEN CostType = 'COS INDIRECT' THEN 'Indirect'
    END AS jobtype,
    CASE WHEN isoverscale = 'Y' THEN 1 ELSE 0 END AS boo_overscale,
    CASE
      WHEN jobcodecategory IS NULL            THEN 'Mechanic'
      WHEN jobcodecategory = 'Probationary'   THEN 'Apprentice'
      WHEN jobcodecategory = 'US SALES REPS'  THEN 'Selling'
      ELSE jobcodecategory
    END AS jobcategory,
    CASE
      WHEN Breakdown2Id = 'TKE101'  THEN 'NI'
      WHEN Breakdown2Id = 'TKE102'  THEN 'MOD'
      WHEN Breakdown2Id = 'TKE1031' THEN 'SERV'
      WHEN Breakdown2Id = 'TKE1032' THEN 'REPAIR'
    END AS LOB,
    branch.alias AS Entity,
    TRUNC(TO_DATE(CAST(dateint AS STRING), 'yyyyMMdd'), 'MONTH') AS `Calendar Date`,
    fte
  FROM `na-cdp-bu-hr-dev`.curated.fte AS fte
  LEFT JOIN branchnames AS branch
    ON branch.backendname = fte.epmbranch
  WHERE CostType IN ('COS DIRECT','COS INDIRECT')
    AND Breakdown2Id IN ('TKE101', 'TKE102', 'TKE1031', 'TKE1032')
    AND fte <> 0
    AND YEAR(TO_DATE(CAST(dateint AS STRING), 'yyyyMMdd')) = calendaryear
    AND MONTH(TO_DATE(CAST(dateint AS STRING), 'yyyyMMdd')) = calendarmonth
),

fte_output AS (
  -- directjobtypebybranch
  SELECT
    CONCAT(jobtype, ' ', jobcategory, ' ', LOB) AS Account,
    `Calendar Date`,
    Entity,
    LOB,
    ROUND(SUM(fte), 3) AS Value,
    'Actuals' AS Scenario,
    'Headcount' AS Source
  FROM alldata
  WHERE jobtype = 'Direct'
  GROUP BY ALL

  UNION ALL

  -- directjobtypebuna (no jobtype filter, matches your original)
  SELECT
    CONCAT(jobtype, ' ', jobcategory, ' ', LOB) AS Account,
    `Calendar Date`,
    'BU North America' AS Entity,
    LOB,
    ROUND(SUM(fte), 3) AS Value,
    'Actuals' AS Scenario,
    'Headcount' AS Source
  FROM alldata
  GROUP BY ALL

  UNION ALL

  -- headcountbybranch
  SELECT
    CONCAT(jobtype, ' HC ', LOB) AS Account,
    `Calendar Date`,
    Entity,
    LOB,
    ROUND(SUM(fte), 3) AS Value,
    'Actuals' AS Scenario,
    'Headcount' AS Source
  FROM alldata
  GROUP BY ALL

  UNION ALL

  -- headcountbuna
  SELECT
    CONCAT(jobtype, ' HC ', LOB) AS Account,
    `Calendar Date`,
    'BU North America' AS Entity,
    LOB,
    ROUND(SUM(fte), 3) AS Value,
    'Actuals' AS Scenario,
    'Headcount' AS Source
  FROM alldata
  GROUP BY ALL

  UNION ALL

  -- directoverscalebybranch
  SELECT
    'Direct Overscale HC' AS Account,
    `Calendar Date`,
    Entity,
    LOB,
    ROUND(SUM(fte), 3) AS Value,
    'Actuals' AS Scenario,
    'Headcount' AS Source
  FROM alldata
  WHERE boo_overscale = 1 AND jobtype = 'Direct'
  GROUP BY ALL

  UNION ALL

  -- directoverscalebuna
  SELECT
    'Direct Overscale HC' AS Account,
    `Calendar Date`,
    'BU North America' AS Entity,
    LOB,
    ROUND(SUM(fte), 3) AS Value,
    'Actuals' AS Scenario,
    'Headcount' AS Source
  FROM alldata
  WHERE boo_overscale = 1 AND jobtype = 'Direct'
  GROUP BY ALL
),

----------------------------------------------------------------
-- Query 2: USD Amounts
----------------------------------------------------------------
usd_data AS (
  SELECT
    TRUNC(TO_DATE(CAST(dateint AS STRING), 'yyyyMMdd'), 'MONTH') AS `Calendar Date`,
    branch.alias AS Entity,
    CASE
      WHEN costtype = 'COS DIRECT'   THEN 'Direct USD'
      WHEN costtype = 'COS INDIRECT'  THEN 'Indirect USD'
      WHEN costtype = 'G&A'           THEN 'G&A USD'
      WHEN costtype = 'SELLING'       THEN 'Selling USD'
    END AS Account,
    CASE
      WHEN Breakdown2Id = 'TKE101'  THEN 'NI'
      WHEN Breakdown2Id = 'TKE102'  THEN 'MOD'
      WHEN Breakdown2Id = 'TKE1031' THEN 'SERV'
      WHEN Breakdown2Id = 'TKE1032' THEN 'REPAIR'
    END AS LOB,
    usd.AmountAnnual AS Value
  FROM `na-cdp-bu-hr-dev`.curated.usdamount AS usd
  LEFT JOIN branchnames AS branch
    ON branch.backendname = usd.epmbranch
  WHERE CostType IN ('COS DIRECT', 'COS INDIRECT', 'SELLING', 'G&A')
    AND Breakdown2Id IN ('TKE101', 'TKE102', 'TKE1031', 'TKE1032')
    AND usd.AmountAnnual <> 0
    AND YEAR(TO_DATE(CAST(dateint AS STRING), 'yyyyMMdd')) = calendaryear
    AND MONTH(TO_DATE(CAST(dateint AS STRING), 'yyyyMMdd')) = calendarmonth
),

usd_output AS (
  SELECT
    Account,
    `Calendar Date`,
    Entity,
    LOB,
    ROUND(SUM(Value), 3) AS Value,
    'Actuals' AS Scenario,
    'Headcount USD' AS Source
  FROM usd_data
  GROUP BY ALL

  UNION ALL

  SELECT
    Account,
    `Calendar Date`,
    'BU North America' AS Entity,
    LOB,
    ROUND(SUM(Value), 3) AS Value,
    'Actuals' AS Scenario,
    'Headcount USD' AS Source
  FROM usd_data
  GROUP BY ALL
),

----------------------------------------------------------------
-- Query 3: Voluntary Turnover
----------------------------------------------------------------
vt_branch AS (
  SELECT
    TRUNC(TO_DATE(CAST(dateint AS STRING), 'yyyyMMdd'), 'MONTH') AS `Calendar Date`,
    branch.alias AS Entity,
    SUM(ftevoluntaryturnover) AS Value
  FROM `NA-CDP-BU-HR-DEV`.curated.voluntaryturnover AS vt
  LEFT JOIN branchnames AS branch
    ON branch.backendname = vt.EPMBranch
  WHERE YEAR(TO_DATE(CAST(dateint AS STRING), 'yyyyMMdd')) = calendaryear
    AND MONTH(TO_DATE(CAST(dateint AS STRING), 'yyyyMMdd')) = calendarmonth
  GROUP BY ALL
),

vol_output AS (
  -- per-entity turnover
  SELECT
    'Voluntary Turnover' AS Account,
    `Calendar Date`,
    Entity,
    'All LOB' AS LOB,
    Value,
    'Actuals' AS Scenario,
    'Voluntary Turnover' AS Source
  FROM vt_branch

  UNION ALL

  -- BU North America total
  SELECT
    'Voluntary Turnover' AS Account,
    `Calendar Date`,
    'BU North America' AS Entity,
    'All LOB' AS LOB,
    SUM(Value) AS Value,
    'Actuals' AS Scenario,
    'Voluntary Turnover' AS Source
  FROM vt_branch
  GROUP BY `Calendar Date`
)

----------------------------------------------------------------
-- FINAL UNION OF ALL THREE
----------------------------------------------------------------
SELECT Account, `Calendar Date`, Entity, LOB, Value, Scenario, Source FROM fte_output
UNION ALL
SELECT Account, `Calendar Date`, Entity, LOB, Value, Scenario, Source FROM usd_output
UNION ALL
SELECT Account, `Calendar Date`, Entity, LOB, Value, Scenario, Source FROM vol_output;